In [0]:
%pip install openai tiktoken
dbutils.library.restartPython()

In [0]:
import math
import time
import random
import unicodedata
from typing import List

import tiktoken
from openai import OpenAI
from pyspark.sql import functions as F
from pyspark.sql import types as T

# -----------------------------
# CONFIG
# -----------------------------
AZURE_ENDPOINT = "https://datascienceroadshow.services.ai.azure.com/openai/v1/"
DEPLOYMENT_NAME = "embed-v-4-0-2"
API_KEY = "key"

OCR_TABLE = "noc_documents_cohere"
TEXT_COL = "final_text"
EMBED_COL = "embedding"

MAX_RETRIES = 6
BASE_SLEEP_SEC = 1.5

MAX_TOKENS = 1200
OVERLAP_RATIO = 0.15

# -----------------------------
# CLIENT INIT
# -----------------------------
client = OpenAI(
    base_url=AZURE_ENDPOINT,
    api_key=API_KEY
)

# tokenizer (safe generic encoding)
enc = tiktoken.get_encoding("cl100k_base")

# -----------------------------
# TEXT CLEANING
# -----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    text = unicodedata.normalize("NFC", text)
    return text.strip()

# -----------------------------
# TOKEN CHUNKING WITH OVERLAP
# -----------------------------
def chunk_by_tokens(
    text: str,
    max_tokens: int = MAX_TOKENS,
    overlap_ratio: float = OVERLAP_RATIO
) -> List[str]:

    tokens = enc.encode(text)
    if not tokens:
        return []

    overlap_tokens = int(max_tokens * overlap_ratio)
    overlap_tokens = max(0, min(overlap_tokens, max_tokens - 1))

    chunks = []
    start = 0
    total_tokens = len(tokens)

    while start < total_tokens:
        end = min(start + max_tokens, total_tokens)
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))

        if end == total_tokens:
            break

        start = end - overlap_tokens

    return chunks

# -----------------------------
# EMBEDDING WITH RETRIES
# -----------------------------
def embed_batch(texts: List[str]) -> List[List[float]]:

    for attempt in range(MAX_RETRIES):
        try:
            response = client.embeddings.create(
                model=DEPLOYMENT_NAME,
                input=texts
            )

            sorted_data = sorted(response.data, key=lambda x: x.index)
            return [d.embedding for d in sorted_data]

        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            sleep_s = BASE_SLEEP_SEC * (2 ** attempt) + random.random()
            print(f"Retry {attempt+1}/{MAX_RETRIES} after error: {e}")
            time.sleep(sleep_s)

# -----------------------------
# POOLING + NORMALIZATION
# -----------------------------
def mean_pool(vectors: List[List[float]]) -> List[float]:
    dim = len(vectors[0])
    pooled = [0.0] * dim

    for v in vectors:
        for i in range(dim):
            pooled[i] += float(v[i])

    n = float(len(vectors))
    return [x / n for x in pooled]

def l2_normalize(v: List[float]) -> List[float]:
    norm = math.sqrt(sum(x * x for x in v))
    if norm == 0:
        return v
    return [x / norm for x in v]

# -----------------------------
# CORE PROCESSOR
# -----------------------------
def process_documents(limit: int = None):

    print("Starting embedding pipeline for table:", OCR_TABLE)

    # Ensure embedding column exists
    try:
        spark.sql(f"ALTER TABLE {OCR_TABLE} ADD COLUMNS ({EMBED_COL} ARRAY<DOUBLE>)")
        print("Embedding column created.")
    except Exception:
        print("Embedding column already exists.")

    # Select only rows without embeddings
    df = (
        spark.table(OCR_TABLE)
             .select("path", TEXT_COL)
             .where(F.col(TEXT_COL).isNotNull())
             .where(F.col(EMBED_COL).isNull())
    )

    if limit:
        df = df.limit(limit)

    rows = df.collect()
    total_docs = len(rows)

    print(f"Documents to embed this run: {total_docs}")

    processed_count = 0
    failed_count = 0

    for idx, r in enumerate(rows, start=1):

        path = r["path"]
        text = clean_text(r[TEXT_COL])

        try:
            chunks = chunk_by_tokens(text)

            if not chunks:
                print(f"[{idx}/{total_docs}] No chunks for: {path}")
                continue

            chunk_vectors = embed_batch(chunks)
            doc_vector = l2_normalize(mean_pool(chunk_vectors))

            update_df = spark.createDataFrame(
                [(path, [float(x) for x in doc_vector])],
                schema=T.StructType([
                    T.StructField("path", T.StringType(), False),
                    T.StructField("embedding", T.ArrayType(T.DoubleType()), True)
                ])
            )

            update_df.createOrReplaceTempView("emb_update_single")

            spark.sql(f"""
            MERGE INTO {OCR_TABLE} t
            USING emb_update_single s
            ON t.path = s.path
            WHEN MATCHED THEN UPDATE SET
              t.{EMBED_COL} = s.embedding
            """)

            processed_count += 1

            print(f"[{idx}/{total_docs}] ✅ Embedded: {path}")
            print(f"Progress: {processed_count}/{total_docs} completed")

        except Exception as e:
            failed_count += 1
            print(f"[{idx}/{total_docs}] ❌ Failed on {path}: {e}")

    print("\n===== RUN SUMMARY =====")
    print("Total documents selected :", total_docs)
    print("Successfully embedded    :", processed_count)
    print("Failed                   :", failed_count)
    print("Run completed safely.")

# -----------------------------
# ORCHESTRATORS
# -----------------------------
def run_embeddings_full_table():
    process_documents(limit=None)

def run_embeddings_limited(row_limit: int):
    process_documents(limit=row_limit)

In [0]:
run_embeddings_limited(10)

In [0]:
run_embeddings_full_table()

com.databricks.backend.common.rpc.DriverStoppedException: Driver down cause: driver state change (exit code: 137)
	at com.databricks.spark.chauffeur.ChauffeurState.processDriverStateChange(ChauffeurState.scala:310)
	at com.databricks.spark.chauffeur.Chauffeur.onDriverStateChange(Chauffeur.scala:2035)
	at com.databricks.spark.chauffeur.Chauffeur.$anonfun$driverStateOpt$1(Chauffeur.scala:240)
	at com.databricks.spark.chauffeur.Chauffeur.$anonfun$driverStateOpt$1$adapted(Chauffeur.scala:240)
	at com.databricks.spark.chauffeur.DriverDaemonMonitorImpl.$anonfun$goToStopped$1(DriverDaemonMonitorImpl.scala:402)
	at com.databricks.spark.chauffeur.DriverDaemonMonitorImpl.$anonfun$goToStopped$1$adapted(DriverDaemonMonitorImpl.scala:402)
	at scala.collection.immutable.List.foreach(List.scala:333)
	at com.databricks.spark.chauffeur.DriverDaemonMonitorImpl.goToStopped(DriverDaemonMonitorImpl.scala:402)
	at com.databricks.spark.chauffeur.DriverDaemonMonitorImpl.monitorDriver(DriverDaemonMonitorImpl.s

In [0]:
%sql
CREATE TABLE noc_documents_cohere
AS SELECT * FROM noc_documents;

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows,
  SUM(CASE WHEN embedding IS NOT NULL THEN 1 ELSE 0 END) AS completed,
  SUM(CASE WHEN embedding IS NULL THEN 1 ELSE 0 END) AS remaining
FROM noc_documents_cohere;